In [1]:
# Phase I: Imports & Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

sns.set_style("whitegrid")
print("All libraries loaded successfully.")

All libraries loaded successfully.


In [3]:
file_path = '/content/Data Warehouse Multiclass.csv'
df = pd.read_csv(file_path)
print(f"Initial raw dataset shape: {df.shape}")

Initial raw dataset shape: (280985, 39)


In [4]:
# Clean duplicate hospital registry rows
initial_len = len(df)
df.drop_duplicates(inplace=True)
print(f"Cleaned duplicate rows: dropped {initial_len - len(df)}. New shape: {df.shape}")

# Handle comma-delimited disease flags safely
if 'disease_flags' in df.columns and df['disease_flags'].apply(lambda x: isinstance(x, str) and ',' in x).any():
    flags_df = df['disease_flags'].str.split(',', expand=True).astype(int)
    flags_df.columns = [f'disease_flag_{i+1}' for i in range(flags_df.shape[1])]
    df = pd.concat([df, flags_df], axis=1)
    df.drop('disease_flags', axis=1, inplace=True)
    print("Processed 'disease_flags' into individual binary indicator columns.")

# Encoding Categorical Features (Ordinal & Nominal)
ordinal_mappings = {
    'age_level': {'Young': 0, 'Middle': 1, 'Mature': 2, 'Senior': 3},
    'bmi_level': {'Underweight': 0, 'Normal': 1, 'Overweight': 2, 'Obese': 3},
    'education_level': {'Primary': 0, 'Secondary': 1, 'Tertiary': 2},
    'stress_level': {'Low': 0, 'Medium': 1, 'High': 2}
}

for col, mapping in ordinal_mappings.items():
    if col in df.columns:
        df[f'{col}_encoded'] = df[col].map(mapping)
        df.drop(col, axis=1, inplace=True)

nominal_cols = ['gender', 'smoking', 'diabetes', 'physical_activity', 'family_history',
                'low_hdl_cholesterol', 'high_ldl_cholesterol', 'high_blood_pressure',
                'sugar_consumption', 'employment_status']
nominal_cols = [c for c in nominal_cols if c in df.columns]

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

# Stratified Splitting & Leakage-Free Scaling
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

X = df.drop(['composite_key', 'source_dataset', 'label', 'sublabel', 'label_encoded'], axis=1, errors='ignore')
y = df['label_encoded']
sublabels = df['sublabel']

X_train, X_test, y_train, y_test, sub_train, sub_test = train_test_split(
    X, y, sublabels, test_size=0.2, random_state=42, stratify=y
)

numerical_cols = X_train.select_dtypes(include=['int64', 'float64', 'bool']).columns.tolist()
binary_cols = [c for c in numerical_cols if 'disease_flag' in c or any(sub in c for sub in ['gender_', 'smoking_', 'diabetes_', 'physical_activity_', 'family_history_', 'low_hdl_', 'high_ldl_', 'high_blood_pressure_', 'sugar_consumption_', 'employment_status_'])]
cols_to_scale = [c for c in numerical_cols if c not in binary_cols]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print(f"X_train shape: {X_train_scaled.shape} | X_test shape: {X_test_scaled.shape}")
print("Phase I completed: Zero data leakage verified.")

Cleaned duplicate rows: dropped 2284. New shape: (278701, 39)
Processed 'disease_flags' into individual binary indicator columns.
X_train shape: (222960, 41) | X_test shape: (55741, 41)
Phase I completed: Zero data leakage verified.


In [5]:
# Phase II & III: Unsupervised Baseline Profiling & Sub-Clinical Validation
normal_class_val = label_encoder.transform(['Normal'])[0]
normal_mask = (y_train == normal_class_val)
X_train_normal = X_train_scaled[normal_mask]
print(f"Healthy normal baseline cohort size: {len(X_train_normal)}")

# 1. Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
iso_forest.fit(X_train_normal)

test_iso_scores = iso_forest.decision_function(X_test_scaled)
test_iso_preds = iso_forest.predict(X_test_scaled)

X_test_results = X_test.copy()
X_test_results['true_label'] = y_test
X_test_results['sublabel'] = sub_test
X_test_results['iso_score'] = test_iso_scores
X_test_results['is_anomaly_iso'] = (test_iso_preds == -1).astype(int)

# 2. One-Class SVM
oc_svm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)
oc_svm.fit(X_train_normal)
oc_preds = oc_svm.predict(X_test_scaled)
X_test_results['is_anomaly_ocsvm'] = (oc_preds == -1).astype(int)

# 3. Deep Autoencoder
minmax_scaler = MinMaxScaler()
X_train_normal_ae = minmax_scaler.fit_transform(X_train_normal)
X_test_ae = minmax_scaler.transform(X_test_scaled)

input_dim = X_train_normal_ae.shape[1]
encoding_dim = max(int(input_dim / 2), 4)

input_layer = Input(shape=(input_dim,))
encoder = Dense(encoding_dim, activation="relu")(input_layer)
decoder = Dense(input_dim, activation="sigmoid")(encoder)

autoencoder = Model(inputs=input_layer, outputs=decoder)
autoencoder.compile(optimizer='adam', loss='mean_squared_error')

history = autoencoder.fit(X_train_normal_ae, X_train_normal_ae,
                          epochs=15, batch_size=256, shuffle=True,
                          validation_split=0.1, verbose=0)

reconstructed_test = autoencoder.predict(X_test_ae)
test_mse = np.mean(np.power(X_test_ae - reconstructed_test, 2), axis=1)

reconstructed_train_normal = autoencoder.predict(X_train_normal_ae)
train_normal_mse = np.mean(np.power(X_train_normal_ae - reconstructed_train_normal, 2), axis=1)
ae_threshold = np.percentile(train_normal_mse, 95)

X_test_results['ae_mse'] = test_mse
X_test_results['is_anomaly_ae'] = (test_mse > ae_threshold).astype(int)

# Early Warning Audit
sublabel_audit = X_test_results.groupby('sublabel').agg(
    count=('iso_score', 'count'),
    mean_iso_score=('iso_score', 'mean'),
    mean_autoencoder_mse=('ae_mse', 'mean')
).reset_index()

print("Early Warning Audit Complete.")
display(sublabel_audit.sort_values(by='mean_iso_score'))

Healthy normal baseline cohort size: 83718
1742/1742 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
Early Warning Audit Complete.


,sublabel,count,mean_iso_score,mean_autoencoder_mse
3,DI_HY,13381,0.003508,0.103051
0,DI,6390,0.021294,0.067401
6,HY,13829,0.028296,0.056994
2,DI_HT_HY,181,0.034055,0.377806
5,HT_HY,216,0.049904,0.368397
1,DI_HT,286,0.056678,0.324731
4,HT,528,0.076490,0.331883
7,N,20930,0.105970,0.003371


In [6]:
# Phase IV: Hybrid Meta-Feature Supervised Classifier
X_train_augmented = X_train_scaled.copy()
X_train_augmented['iso_score'] = iso_forest.decision_function(X_train_scaled)

X_test_augmented = X_test_scaled.copy()
X_test_augmented['iso_score'] = test_iso_scores

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf_clf.fit(X_train_augmented, y_train)

y_preds = rf_clf.predict(X_test_augmented)
print("\nSupervised Classification Report (Meta-Feature Augmented):")
print(classification_report(y_test, y_preds, target_names=label_encoder.classes_))


Supervised Classification Report (Meta-Feature Augmented):
              precision    recall  f1-score   support

    Abnormal       1.00      1.00      1.00     34811
      Normal       1.00      1.00      1.00     20930

    accuracy                           1.00     55741
   macro avg       1.00      1.00      1.00     55741
weighted avg       1.00      1.00      1.00     55741

